# Сравнение входов TF-IDF

Те же четыре текстовых входа, сплит и 1000 тестовых контекстов, что в `BM25_pipeline.ipynb`. Для каждого входа сравниваются слова и символьные 3–5-граммы внутри слов (`char_wb`). Итого восемь вариантов. Корпус включает уникальные объявления всего train, в том числе объявления отложенных взаимодействий; для каждого item_id берём последнюю строку файла.

Оба режима используют один интерфейс `TfidfRetriever`, одинаковый предел словаря в 20 000 признаков, `min_df=2` и первые 100 символов каждого текста объявления. Эти ограничения снижают стоимость символьного индекса; длинные описания и параметры здесь усечены для обоих режимов. Поэтому прямое числовое сравнение с BM25 на полных текстах некорректно. В объединённом поле усечение после склейки оставляет прежде всего заголовок и начало описания. Значения можно увеличить при большем объёме памяти. Словесный режим использует `RussianTokenizer` без лемматизации, как BM25. Ранжирование — косинусное сходство TF-IDF-векторов.

Recall@10 — средняя по контекстам доля найденных уникальных объявлений из наблюдаемых пар. Пустой запрос получает 0. Это восстановление наблюдаемых пар, а не оценка по полной разметке релевантности.

In [1]:
from pathlib import Path
import sys
import gc

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/split.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.split import split_queries
from src.retrievers.tfidf import TfidfRetriever

SEED = 42
K = 10
N_TEST_QUERIES = 1000
MAX_FEATURES = 20_000
MAX_CHARS = 100
MIN_DF = 2
QUERY_COLUMNS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
]
ITEM_FIELDS = {
    "Заголовок": "item_title_raw",
    "Параметры": "item_infm_params_text",
    "Описание": "item_description_raw",
    "Заголовок + описание": "item_text",
}
results = []

## Данные и сплит

In [2]:
data = pl.read_parquet(ROOT / "data/train.parquet", columns=QUERY_COLUMNS + ["item_id"])
data = data.with_columns(pl.Series("split", split_queries(data["search_query"], seed=SEED)))
test = (
    data.filter(pl.col("split") == "test")
    .group_by(QUERY_COLUMNS).agg(pl.col("item_id").unique().alias("targets"))
    .sort(QUERY_COLUMNS)
)
test = test.sample(n=min(N_TEST_QUERIES, len(test)), seed=SEED, shuffle=True)
queries = pd.DataFrame(test.select(QUERY_COLUMNS).to_dict(as_series=False))
targets = [set(ids) for ids in test["targets"].to_list()]
assert all(targets)
print(f"Тестовая выборка: {len(queries):,} контекстов")
display(data.group_by("split").len().sort("split"))
del data, test
gc.collect();

Тестовая выборка: 1,000 контекстов


split,len
str,u32
"""test""",83218
"""train""",360319
"""val""",54136


## Recall@K и запуск

Каждый корпус читается по нужным колонкам, индекс удаляется после оценки. Это позволяет прогнать длинные поля при ограниченной памяти. Порядок item_id одинаков для всех вариантов.

In [3]:
def recall_at_k(candidates, targets, k):
    retrieved = (
        candidates.loc[candidates["rank"] <= k]
        .groupby("query_index")["item_id"].agg(set).to_dict()
    )
    return float(np.mean([
        len(relevant & retrieved.get(i, set())) / len(relevant)
        for i, relevant in enumerate(targets)
    ]))

# Та же проверка определения метрики, что в BM25_pipeline.ipynb.
example = pd.DataFrame({"query_index": [0, 0, 0], "item_id": ["a", "a", "b"], "rank": [1, 2, 3]})
assert recall_at_k(example, [{"a", "b"}, {"c"}], k=2) == 0.25
assert recall_at_k(example.iloc[:0], [{"a"}], k=2) == 0.0

def run_experiment(name, query_column, item_column, analyzer):
    scan = pl.scan_parquet(ROOT / "data/train.parquet")
    if item_column == "item_text":
        text = pl.concat_str([
            pl.col("item_title_raw").fill_null(""),
            pl.col("item_description_raw").fill_null(""),
        ], separator=" ").alias(item_column)
    else:
        text = pl.col(item_column)
    text = text.str.slice(0, MAX_CHARS)
    items = (
        scan.select("item_id", text)
        .unique(subset="item_id", keep="last", maintain_order=True)
        .sort("item_id").collect()
    )
    corpus = pd.DataFrame(items.to_dict(as_series=False))
    model = TfidfRetriever(
        item_text_column=item_column,
        query_text_column=query_column,
        analyzer=analyzer,
        ngram_range=(1, 1) if analyzer == "word" else (3, 5),
        min_df=MIN_DF,
        max_features=MAX_FEATURES,
        batch_size=16,
    ).fit(corpus)
    candidates = model.retrieve(queries, k=K)
    result = {
        "Вход": name, "Токенизация": analyzer,
        f"Recall@{K}": recall_at_k(candidates, targets, K),
        "Объявлений": len(corpus),
        "Признаков": len(model.vectorizer_.vocabulary_),
        "Запросов с выдачей": candidates.query_index.nunique(),
    }
    print(result)
    del model, corpus, items, candidates
    gc.collect()
    return result

## Запрос → заголовок

### Слова

In [4]:
results.append(run_experiment('Запрос → заголовок', 'search_query', 'item_title_raw', 'word'))

{'Вход': 'Запрос → заголовок', 'Токенизация': 'word', 'Recall@10': 0.05433333333333333, 'Объявлений': 344825, 'Признаков': 20000, 'Запросов с выдачей': 994}


### Символьные 3–5-граммы

In [5]:
results.append(run_experiment('Запрос → заголовок', 'search_query', 'item_title_raw', 'char_wb'))

{'Вход': 'Запрос → заголовок', 'Токенизация': 'char_wb', 'Recall@10': 0.05475, 'Объявлений': 344825, 'Признаков': 20000, 'Запросов с выдачей': 1000}


## Параметры запроса → параметры объявления

Пустые параметры запроса остаются в оценке: если совпадений нет, Recall для контекста равен 0.

### Слова

In [6]:
results.append(run_experiment('Параметры запроса → параметры объявления', 'search_infm_params_text', 'item_infm_params_text', 'word'))

{'Вход': 'Параметры запроса → параметры объявления', 'Токенизация': 'word', 'Recall@10': 0.002, 'Объявлений': 344825, 'Признаков': 17472, 'Запросов с выдачей': 641}


### Символьные 3–5-граммы

In [7]:
results.append(run_experiment('Параметры запроса → параметры объявления', 'search_infm_params_text', 'item_infm_params_text', 'char_wb'))

{'Вход': 'Параметры запроса → параметры объявления', 'Токенизация': 'char_wb', 'Recall@10': 0.001, 'Объявлений': 344825, 'Признаков': 20000, 'Запросов с выдачей': 641}


## Запрос → описание

### Слова

In [8]:
results.append(run_experiment('Запрос → описание', 'search_query', 'item_description_raw', 'word'))

{'Вход': 'Запрос → описание', 'Токенизация': 'word', 'Recall@10': 0.028293478260869566, 'Объявлений': 344825, 'Признаков': 20000, 'Запросов с выдачей': 984}


### Символьные 3–5-граммы

In [9]:
results.append(run_experiment('Запрос → описание', 'search_query', 'item_description_raw', 'char_wb'))

{'Вход': 'Запрос → описание', 'Токенизация': 'char_wb', 'Recall@10': 0.028210144927536235, 'Объявлений': 344825, 'Признаков': 20000, 'Запросов с выдачей': 999}


## Запрос → заголовок + описание

### Слова

In [10]:
results.append(run_experiment('Запрос → заголовок + описание', 'search_query', 'item_text', 'word'))

{'Вход': 'Запрос → заголовок + описание', 'Токенизация': 'word', 'Recall@10': 0.048083333333333325, 'Объявлений': 344825, 'Признаков': 20000, 'Запросов с выдачей': 987}


### Символьные 3–5-граммы

In [11]:
results.append(run_experiment('Запрос → заголовок + описание', 'search_query', 'item_text', 'char_wb'))

{'Вход': 'Запрос → заголовок + описание', 'Токенизация': 'char_wb', 'Recall@10': 0.052210144927536235, 'Объявлений': 344825, 'Признаков': 20000, 'Запросов с выдачей': 999}


## Сравнение

Обе токенизации каждого входа оценены на тех же контекстах и с тем же K. Это одиночные ретриверы: объединение их выдач требует отдельного эксперимента.

In [12]:
comparison = pd.DataFrame(results)
display(comparison.sort_values(f"Recall@{K}", ascending=False).reset_index(drop=True))
display(comparison.pivot(index="Вход", columns="Токенизация", values=f"Recall@{K}"))

,Вход,Токенизация,Recall@10,Объявлений,Признаков,Запросов с выдачей
0,Запрос → заголовок,char_wb,0.054750,344825,20000,1000
1,Запрос → заголовок,word,0.054333,344825,20000,994
2,Запрос → заголовок + описание,char_wb,0.052210,344825,20000,999
3,Запрос → заголовок + описание,word,0.048083,344825,20000,987
4,Запрос → описание,word,0.028293,344825,20000,984
5,Запрос → описание,char_wb,0.028210,344825,20000,999
6,Параметры запроса → параметры объявления,word,0.002000,344825,17472,641
7,Параметры запроса → параметры объявления,char_wb,0.001000,344825,20000,641


Токенизация,char_wb,word
Вход,,
Запрос → заголовок,0.05475,0.054333
Запрос → заголовок + описание,0.05221,0.048083
Запрос → описание,0.02821,0.028293
Параметры запроса → параметры объявления,0.00100,0.002000
